In [1]:
from analysis.processors.base import BaseProcessor
from coffea.nanoevents import NanoAODSchema
from coffea import processor
import warnings
NanoAODSchema.warn_missing_crossrefs = False

workflow = "2b1e"
year = "2022preEE"

fileset = {'2022preEE': {
     'TTTo2L2Nu_1': {
         'files': {'root://cms-xrd-global.cern.ch:1094//store/mc/Run3Summer22NanoAODv12/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/50000/d51aa7d0-59ab-4ce7-8852-3f9a1ec14bc6.root': 'Events'},
         'metadata': {'short_name': 'TTTo2L2Nu'}},
    'SingleMuonC_1': {
       'files': {"root://cms-xrd-global.cern.ch:1094//store/data/Run2022C/DoubleMuon/NANOAOD/22Sep2023-v1/50000/9f317280-5380-49f7-8a30-3e3bcd48dc9c.root": "Events"},
       'metadata': {'short_name': 'SingleMuon'}}
}
          }

In [2]:
!voms-proxy-init --voms cms

Contacting voms2-cms-auth.cern.ch:443 [/DC=ch/DC=cern/OU=computers/CN=voms2-cms-auth.cern.ch] "cms"...
Error contacting voms2-cms-auth.cern.ch:443 for VO cms: voms2-cms-auth.cern.ch
Error contacting voms2-cms-auth.cern.ch:443 for VO cms: voms2-cms-auth.cern.ch
Error contacting voms2-cms-auth.cern.ch:443 for VO cms: REST and legacy VOMS endpoints failed.
Contacting voms-cms-auth.cern.ch:443 [/DC=ch/DC=cern/OU=computers/CN=cms-auth.cern.ch] "cms"...
Remote VOMS server contacted succesfully.


Created proxy in /tmp/x509up_u182284.

Your proxy is valid until Tue Sep 01 09:23:00 CEST 2026


In [2]:
futures_run = processor.Runner(
    executor=processor.FuturesExecutor(workers=4, compression=None),
    schema=NanoAODSchema,
    savemetrics=False,
)
out = futures_run(fileset[year], treename="Events", processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"))
out["metadata"]

Output()

Output()

There are 93447 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 91185 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 20773 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 20776 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 115 nan entries in the corrected pt. This might be due to the number of tracker layers hitting boundaries. Setting those entries to their initial value.
There are 81696 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.
There are 90851 events with muon pt outside of [26,200] GeV. Setting those entries to their initial value.


{'sumw': np.float32(3.791133e+06),
 'base': {'cutflow': {'initial': np.float32(3.791133e+06),
   'goodvertex': np.float32(3.754325e+06),
   'lumi': np.float32(3.68228e+06),
   'trigger': np.float32(1.0580961e+06),
   'trigger_match': np.float32(1.0473634e+06),
   'metfilters': np.float32(1.04687675e+06),
   'hemcleaning': np.float32(1.04687675e+06),
   'met_50': np.float32(743549.25),
   'exactly_two_bjets': np.float32(280270.94),
   'tau_veto': np.float32(280270.94),
   'muon_veto': np.float32(229972.53),
   'exactly_one_electron': np.float32(106829.83)},
  'weighted_final_nevents': np.float64(95204.14578667234)}}

In [9]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
filename = list(fileset[year]['TTTo2L2Nu_1']['files'].keys())[0]
events = NanoEventsFactory.from_root(
    filename,
    treepath="Events",
    entry_stop=1_000,
    metadata={"dataset": "TTTo2L2Nu"},
    schemaclass=NanoAODSchema,
    mode="virtual",
).events()

### Why HTCondor doesn't works in SWAN?

In [ ]:
from dask.distributed import Client
client = Client("tls://10.100.197.122:30795")
dask_run = processor.Runner( #this is new runner function for coffea 2025.10
    executor=processor.DaskExecutor(client=client, compression=None), #execute via dask workers
    schema=NanoAODSchema,
    chunksize=100_000,
    skipbadfiles=False,
    savemetrics=False,
)
#histograms = dask_run(test_fileset, processor_instance=MuonProcessor(workflow_path, year))
histograms = dask_run(fileset[year], treename="Events", processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"))